In [1]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Q1

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('de-zoomcamp-spark') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 11:01:24 WARN Utils: Your hostname, DESKTOP-2MHI7LI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/10 11:01:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 11:01:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


# Q2

In [4]:
taxi_df=spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [5]:
taxi_df.head(1)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75)]

In [6]:
!rm -r q2

rm: cannot remove 'q2': No such file or directory


In [7]:
taxi_df.repartition(4).write.parquet('q2')

In [8]:
!ls -alh q2

total 99M
drwxr-xr-x 2 mahdi mahdi 4.0K Mar 10 11:01 .
drwxr-xr-x 5 mahdi mahdi 4.0K Mar 10 11:01 ..
-rw-r--r-- 1 mahdi mahdi    8 Mar 10 11:01 ._SUCCESS.crc
-rw-r--r-- 1 mahdi mahdi 196K Mar 10 11:01 .part-00000-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.parquet.crc
-rw-r--r-- 1 mahdi mahdi 196K Mar 10 11:01 .part-00001-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.parquet.crc
-rw-r--r-- 1 mahdi mahdi 196K Mar 10 11:01 .part-00002-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.parquet.crc
-rw-r--r-- 1 mahdi mahdi 196K Mar 10 11:01 .part-00003-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.parquet.crc
-rw-r--r-- 1 mahdi mahdi    0 Mar 10 11:01 _SUCCESS
-rw-r--r-- 1 mahdi mahdi  25M Mar 10 11:01 part-00000-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.parquet
-rw-r--r-- 1 mahdi mahdi  25M Mar 10 11:01 part-00001-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.parquet
-rw-r--r-- 1 mahdi mahdi  25M Mar 10 11:01 part-00002-693bd1fc-2ec6-4b5c-80af-b1bee3a41d38-c000.snappy.p

# Q3

In [9]:
taxi_df.filter(F.to_date(taxi_df['tpep_pickup_datetime'])=='2025-11-15').count()

162604

# Q4

In [10]:
taxi_df=taxi_df.withColumn("duration", taxi_df['tpep_dropoff_datetime']-taxi_df['tpep_pickup_datetime'])

In [11]:
max_duration = taxi_df.agg(F.max(taxi_df["duration"])).collect()[0][0]

In [12]:
max_duration.total_seconds()/3600

90.64666666666666

# Q6

In [13]:
taxi_zone_lookup_df=spark.read.csv('taxi_zone_lookup.csv', header=True)

In [14]:
taxi_zone_lookup_df = taxi_zone_lookup_df.withColumn("LocationID", taxi_zone_lookup_df["LocationID"].cast("integer"))

In [15]:
taxi_zone_lookup_df.schema

StructType([StructField('LocationID', IntegerType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [16]:
taxi_df.head(1)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75, duration=datetime.timedelta(0))]

In [17]:
taxi_pu_count_df=taxi_df.groupby('PULocationID').count().orderBy('count')
taxi_pu_count_df=taxi_pu_count_df.withColumnRenamed('PULocationID','LocationID')

In [18]:
taxi_pu_count_df.join(taxi_zone_lookup_df, on="LocationID", how="left").orderBy('count').show()

+----------+-----+-------------+--------------------+------------+
|LocationID|count|      Borough|                Zone|service_zone|
+----------+-----+-------------+--------------------+------------+
|       105|    1|    Manhattan|Governor's Island...| Yellow Zone|
|         5|    1|Staten Island|       Arden Heights|   Boro Zone|
|        84|    1|Staten Island|Eltingville/Annad...|   Boro Zone|
|       187|    3|Staten Island|       Port Richmond|   Boro Zone|
|       204|    4|Staten Island|   Rossville/Woodrow|   Boro Zone|
|       199|    4|        Bronx|       Rikers Island|   Boro Zone|
|       111|    4|     Brooklyn| Green-Wood Cemetery|   Boro Zone|
|       109|    4|Staten Island|         Great Kills|   Boro Zone|
|         2|    5|       Queens|         Jamaica Bay|   Boro Zone|
|       251|   12|Staten Island|         Westerleigh|   Boro Zone|
|        59|   14|        Bronx|        Crotona Park|   Boro Zone|
|       176|   14|Staten Island|             Oakwood|   Boro Z